In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Job interview agent


In [3]:
interview_agent = Agent(
    name="Job Interview Coach",
    instructions="""
    Help users practice for job interviews. Ask one interview question at a time.
    After each answer, give concise, constructive feedback and suggest one way
    to improve it. Adapt questions to the role the user is applying for.
    """
)

response_stream = Runner.run_streamed(
    interview_agent,
    "I’m applying for a software developer role. Ask me an interview question."
)

In [4]:
with trace("Job Interview Coach"):
    result = await Runner.run(
        interview_agent,
        "I'm applying for a software developer role. Ask me one interview question."
    )

print(result.final_output)

Can you describe a performance bottleneck you identified in a project, the steps you took to fix it, and the impact of your changes?


In [5]:
response_stream = Runner.run_streamed(
    interview_agent,
    "I'm applying for a software developer role. Ask me one interview question."
)

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Question: Given an array of integers and a target sum, implement an algorithm to find two numbers that add up to the target in O(n) time. Explain your approach and discuss space-time trade-offs.